# 02 - PPG Preprocessing

Handle missing samples, smooth and normalize each PPG window, remove unusable windows, and save the clean Delta table.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

clean_schema = T.StructType([
    T.StructField('clean_signal', T.ArrayType(T.DoubleType()), True),
    T.StructField('normalized_signal', T.ArrayType(T.DoubleType()), True),
    T.StructField('missing_fraction', T.DoubleType(), True),
    T.StructField('signal_std', T.DoubleType(), True),
    T.StructField('is_usable', T.BooleanType(), True),
    T.StructField('reject_reason', T.StringType(), True),
])

@F.udf(clean_schema)
def clean_ppg_window(ppg_values):
    import numpy as np

    tokens = [] if ppg_values is None else str(ppg_values).split(';')
    values = []
    for token in tokens:
        token = token.strip()
        if token == '' or token.lower() in {'nan', 'none', 'null'}:
            values.append(np.nan)
        else:
            values.append(float(token))

    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return ([], [], 1.0, 0.0, False, 'empty_signal')

    missing_fraction = float(np.mean(~np.isfinite(arr)))
    valid_mask = np.isfinite(arr)
    if valid_mask.any() and not valid_mask.all():
        x = np.arange(arr.size)
        arr[~valid_mask] = np.interp(x[~valid_mask], x[valid_mask], arr[valid_mask])
    elif not valid_mask.any():
        arr = np.zeros_like(arr)

    signal_std = float(np.std(arr))
    window_size = min(5, arr.size)
    if window_size > 1:
        kernel = np.ones(window_size, dtype=float) / window_size
        pad_left = window_size // 2
        pad_right = window_size - 1 - pad_left
        smoothed = np.convolve(np.pad(arr, (pad_left, pad_right), mode='edge'), kernel, mode='valid')
    else:
        smoothed = arr

    smooth_std = float(np.std(smoothed))
    if smooth_std < 1e-8:
        normalized = np.zeros_like(smoothed)
    else:
        normalized = (smoothed - float(np.mean(smoothed))) / smooth_std

    is_usable = missing_fraction <= 0.20 and signal_std >= 0.02
    if missing_fraction > 0.20:
        reject_reason = 'too_many_missing_samples'
    elif signal_std < 0.02:
        reject_reason = 'low_variance_signal'
    else:
        reject_reason = 'accepted'

    return ([float(x) for x in smoothed], [float(x) for x in normalized], missing_fraction, signal_std, bool(is_usable), reject_reason)

raw_df = spark.table('cardio_ppg_raw')

with_clean_result = raw_df.withColumn('clean_result', clean_ppg_window(F.col('ppg_values')))
preprocessed_df = with_clean_result.select(
    '*',
    F.col('clean_result.clean_signal').alias('clean_signal'),
    F.col('clean_result.normalized_signal').alias('normalized_signal'),
    F.col('clean_result.missing_fraction').alias('missing_fraction'),
    F.col('clean_result.signal_std').alias('signal_std'),
    F.col('clean_result.is_usable').alias('is_usable'),
    F.col('clean_result.reject_reason').alias('reject_reason'),
).drop('clean_result')

display(preprocessed_df.groupBy('is_usable', 'reject_reason').count())

clean_df = preprocessed_df.filter(F.col('is_usable') == True)
clean_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cardio_ppg_clean')

print('Saved Delta table: cardio_ppg_clean')
print(f'Clean rows: {clean_df.count()}')
display(clean_df.select('patient_id', 'window_id', 'missing_fraction', 'signal_std', 'normalized_signal').limit(10))
